# Parallel step 49 — identify candidate events from existing picks

Run this notebook before 50a/50b. It creates diagnostic proposals only: no existing catalog, pick, waveform, or origin is changed.

P picks are clustered in time, with a maximum cluster span to prevent chains linking many arrivals. Station counts are independent station identities, not channel counts. Repeated station arrivals, disagreement with historical groupings, sensitivity to the clustering gap, and weak waveform support are flagged. S picks are assigned only when their historical group identifies one cluster and a same-station P precedes them within the configured allowance. Everything else remains unresolved.

Waveform support uses a smoothed RMS envelope around the **existing pick**, compared with a preceding local baseline. This is a diagnostic, not an autopicker or calibrated earthquake detector. No new arrivals or events lacking P picks are detected. A supported candidate requires at least two supporting stations, stable clustering, and no flagged waveform/channel problems. Origin associations must be mutually unique within the configured origin-to-first-P interval. Ambiguous alternatives are retained, not resolved greedily.

The default checks a reproducible sample of 150 picked waveform windows while clustering all existing picks. Unchecked windows cannot enable automatic origin associations. Set `max_waveforms=None` for full waveform checking. Initial thresholds are provisional and should be reviewed on a small sample before manuscript use.

All products go to a new trial directory. Each run refuses a nonempty destination.

In [ ]:
from pathlib import Path
import os, sys
from IPython.display import display

# Works from this subdirectory, 10_conversion, or the repository root.
roots = [Path.cwd(), *Path.cwd().parents]
if os.environ.get('PINATUBO_CODE_ROOT'):
    roots.insert(0, Path(os.environ['PINATUBO_CODE_ROOT']))
candidates = [p for r in roots for p in (r, r / '10_conversion')]
PIPELINE_DIR = next(p for p in candidates if (p / 'lib' / 'clock.py').is_file())
CODE_DIR = next(p for p in [Path.cwd(), PIPELINE_DIR / 'catalog_parallel']
                if (p / 'catalog_workflow.py').is_file())
sys.path.insert(0, str(PIPELINE_DIR))
sys.path.insert(0, str(CODE_DIR))
from lib.config import pipeline_paths
from catalog_workflow import build_sources, merge_sources
P = pipeline_paths()

# Both notebooks must use the same trial root. Choose a NEW name for a fresh run.
RUN_NAME = 'candidate_trial_01'
TRIAL_ROOT = Path(os.environ.get('PINATUBO_CATALOG_TRIAL_ROOT',
                 str(Path(P['OUTPUT_ROOT']) / 'catalog_parallel' / RUN_NAME)))
SOURCE_DIR = TRIAL_ROOT / 'sources'
MERGED_DIR = TRIAL_ROOT / 'merged'
print('Existing inputs:', P['OUTPUT_ROOT'])
print('Individual catalogs:', SOURCE_DIR)
print('Merged catalog:', MERGED_DIR)

CANDIDATE_DIR = TRIAL_ROOT / 'candidates'


In [ ]:
from candidate_events import Settings, run as identify_candidates

CANDIDATE_SETTINGS = Settings(
    gap_s=2.0, max_span_s=6.0,
    sensitivity_gaps_s=(1.5, 2.0, 3.0),
    min_stations=2, min_energy_ratio=3.0,
    max_waveforms=150,  # None checks ALL picked waveform windows.
    max_plots=30, sample_seed=1991,
)
display(CANDIDATE_SETTINGS)

In [ ]:
diagnostics = identify_candidates(P, CANDIDATE_DIR, CANDIDATE_SETTINGS)

In [ ]:
import pandas as pd
from IPython.display import Image
print({k:v for k,v in diagnostics.items() if k.endswith('_count') or k.endswith('_counts') or k in ('waveforms_read','waveform_errors','plots')})
candidates = pd.read_csv(CANDIDATE_DIR / 'candidates.csv')
review = pd.read_csv(CANDIDATE_DIR / 'review_queue.csv')
associations = pd.read_csv(CANDIDATE_DIR / 'origin_associations.csv')
display(candidates.head(10))
audit = pd.read_csv(CANDIDATE_DIR / 'waveform_audit.csv')
checked = set(audit.loc[audit.status.eq('read'), 'window_id'])
display(review[review.window_id.isin(checked)].head(10))
display(associations[associations.status.eq('supported_unique_proposal')].head(10))
for image in sorted((CANDIDATE_DIR / 'plots').glob('*.png'))[:3]:
    display(Image(filename=str(image)))

## Review before adoption

Inspect both straightforward and difficult examples. The CSVs include complete pick accounting, candidate memberships, historical group IDs, waveform support and quality, parameter sensitivity, every plausible temporal origin edge, and unresolved decisions. Plots show waveform channels, P/S markers, normalized envelopes, candidate bands, and origin times.

Copy `review_queue.csv` to a separate working review file before editing it: the diagnostic bundle is fingerprinted and downstream integration refuses edited input products. Human review decisions are documentation only at this stage; no overrides are automatically applied.

Running 50a with `USE_CANDIDATE_ASSOCIATIONS=True` applies only supported, mutually unique origin proposals in a separate trial catalog. Every other historical origin group stays separate and retains its candidate alternatives as provenance. All picks, including previously unmapped picks, are retained as assigned or unresolved observations. 50b verifies the saved catalogs. This does not constitute validation of the earthquake identifications.